# ベースライン拡張（未使用特徴量を追加してスコア比較）

- **元ファイル**: `train_baseline.ipynb` は触らず、戻りたいときの基準として残す。
- **このノート**: ベースラインと同じパイプラインに加え、**まだ使っていない列**を何かしらの処理で特徴量として追加。
- **比較**: ベース（FEATURES のみ）の CV AUC と、拡張（ベース＋未使用列）の CV AUC を表示。
- 新しい特徴量・前処理を試すときはこのファイルで行う。

In [1]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from preprocess import load_train_test
from feature_engineering import create_features, FEATURES

print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [3]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

Train: 653,507, Test: 40,716


In [4]:
train = create_features(train)
test = create_features(test)
print("特徴量作成完了.")

特徴量作成完了.


## 未使用列の取得と処理

- **拡張で追加する列**: ベース（FEATURES）・識別子・目的変数・元の日付・ジャンル文字列を除いた「未使用」列。
  例: rotten_tomatoes_link, critic_name, movie_title, movie_info, directors, authors, actors, production_company など。
- **target は特徴量に含めていません**（特徴量に使わないのは target のみという想定で除外）。
- original_release_date は create_features で review_date と同じく年・月・曜日に分解（release_year, release_month, release_dayofweek）して FEATURES で使用。

In [5]:
# 特徴量に使わないのは target のみという想定（ID は識別子のため除外）
EXCLUDE_FROM_FEATURES = {"target", "ID"}
# 以下は create_features で派生済みのため、未使用列の候補から外す
SOURCE_COLUMNS_ALREADY_DERIVED = {"review_date", "original_release_date", "genres"}

common = [c for c in train.columns if c in test.columns]
unused = [c for c in common if c not in FEATURES and c not in EXCLUDE_FROM_FEATURES and c not in SOURCE_COLUMNS_ALREADY_DERIVED]

print(f"拡張で追加する未使用列: {len(unused)}個")
for c in unused:
    print(f"  - {c}")

拡張で追加する未使用列: 8個
  - rotten_tomatoes_link
  - critic_name
  - movie_title
  - movie_info
  - directors
  - authors
  - actors
  - production_company


In [6]:
# 未使用列をモデルに入れられる形に変換（train の統計で test も埋める）
train_work = train.copy()
test_work = test.copy()
extra_cols = []

for col in unused:
    if col not in train_work.columns or col not in test_work.columns:
        continue
    tr_ser = train_work[col]
    te_ser = test_work[col]
    if pd.api.types.is_numeric_dtype(tr_ser):
        med = tr_ser.median()
        train_work[col] = tr_ser.fillna(med)
        test_work[col] = te_ser.fillna(med)
        extra_cols.append(col)
    else:
        train_work[col] = tr_ser.fillna("missing").astype("category")
        test_work[col] = te_ser.fillna("missing").astype("category")
        extra_cols.append(col)

FEATURES_BASELINE = list(FEATURES)
FEATURES_EXTENDED = FEATURES_BASELINE + extra_cols
print(f"ベース特徴量: {len(FEATURES_BASELINE)}個 / 拡張後: {len(FEATURES_EXTENDED)}個")

ベース特徴量: 19個 / 拡張後: 27個


In [7]:
# 時系列CVの分割（train_baseline と同じ）
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train_work["review_year"] < vy)[0]
    val_idx = np.where(train_work["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
for i, (_, val_idx) in enumerate(time_splits):
    print(f"  Fold{i+1}: val_year={VAL_YEARS[i]}, val_n={len(val_idx):,}")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
  Fold1: val_year=2013, val_n=47,263
  Fold2: val_year=2014, val_n=45,165
  Fold3: val_year=2015, val_n=49,842
  Fold4: val_year=2016, val_n=36,455


In [8]:
y = train_work["target"].values
lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}

### ① ベース（FEATURES のみ）で CV

In [9]:
X_base = train_work[FEATURES_BASELINE]
X_test_base = test_work[FEATURES_BASELINE]

fold_scores_base = []
for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr, X_val = X_base.iloc[tr_idx], X_base.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    pred_val = model.predict_proba(X_val)[:, 1]
    fold_scores_base.append(roc_auc_score(y_val, pred_val))

cv_auc_base = np.mean(fold_scores_base)
print(f"ベース（FEATURES のみ） CV AUC (fold mean): {cv_auc_base:.4f} ± {np.std(fold_scores_base):.4f}")

ベース（FEATURES のみ） CV AUC (fold mean): 0.6923 ± 0.0094


### ② 拡張（ベース＋未使用列）で CV

In [10]:
X_ext = train_work[FEATURES_EXTENDED]
X_test_ext = test_work[FEATURES_EXTENDED]

fold_scores_ext = []
for fold, (tr_idx, val_idx) in enumerate(time_splits):
    X_tr, X_val = X_ext.iloc[tr_idx], X_ext.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
    pred_val = model.predict_proba(X_val)[:, 1]
    fold_scores_ext.append(roc_auc_score(y_val, pred_val))

cv_auc_ext = np.mean(fold_scores_ext)
print(f"拡張（ベース+未使用列） CV AUC (fold mean): {cv_auc_ext:.4f} ± {np.std(fold_scores_ext):.4f}")

拡張（ベース+未使用列） CV AUC (fold mean): 0.7356 ± 0.0057


In [12]:
# ベース vs 拡張 スコア比較
results = pd.DataFrame([
    {"設定": "ベース（FEATURESのみ）", "n_features": len(FEATURES_BASELINE), "CV_AUC": round(cv_auc_base, 4)},
    {"設定": "拡張（ベース+未使用列）", "n_features": len(FEATURES_EXTENDED), "CV_AUC": round(cv_auc_ext, 4)},
])
display(results)
print(f"差分: {cv_auc_ext - cv_auc_base:+.4f}")

,設定,n_features,CV_AUC
0,ベース（FEATURESのみ）,19,0.6923
1,拡張（ベース+未使用列）,27,0.7356


差分: +0.0433


## 前処理を1つずつ変えた実験（どの特徴量の前処理が効くか）

- **ベース**: 拡張（FEATURES + 8列を category のまま）の CV を基準とする。上記「② 拡張」の CV と同一設定なので、1 行目の CV_AUC は拡張の値と一致する想定。
- **実験**: 前処理を **1 種類ずつ** だけ変えた設定で CV を回し、ベースからどれだけ伸びたか（**差分(ベース比)**）を比較する。
- **目的**: どの特徴量の前処理が効いているかを確認する。

In [ ]:
# 前処理実験の定義: ベース + 1 列だけ前処理を変えた設定
# kind: "meta" = 列をやめて長さ・語数のみ, "freq_only" = 列をやめて頻度のみ, "add_freq" = category は残して頻度を追加
PREPROCESS_EXPERIMENTS = [
    {"name": "ベース（8列はcategory）", "col": None, "kind": None},
    {"name": "movie_info→長さ・語数のみ", "col": "movie_info", "kind": "meta"},
    {"name": "movie_title→頻度のみ", "col": "movie_title", "kind": "freq_only"},
    {"name": "critic_name+頻度追加", "col": "critic_name", "kind": "add_freq"},
    {"name": "directors+頻度追加", "col": "directors", "kind": "add_freq"},
    {"name": "authors+頻度追加", "col": "authors", "kind": "add_freq"},
    {"name": "actors+頻度追加", "col": "actors", "kind": "add_freq"},
    {"name": "rotten_tomatoes_link+頻度追加", "col": "rotten_tomatoes_link", "kind": "add_freq"},
    {"name": "production_company+頻度追加", "col": "production_company", "kind": "add_freq"},
]


def build_X_for_config(train_df, test_df, config, tr_idx=None, val_idx=None):
    """
    config に従って特徴量行列を組み立てる。
    tr_idx, val_idx が渡されていれば CV 用（freq は tr のみで計算）。なければ全 train / test 用。
    戻り値: (X_tr, X_val, feature_list) または (X_train, X_test, feature_list)
    """
    base_feats = FEATURES_BASELINE + [c for c in extra_cols if c in train_df.columns]
    col, kind = config.get("col"), config.get("kind")

    if col is None or kind is None:
        if tr_idx is not None:
            return (
                train_df.iloc[tr_idx][base_feats].copy(),
                train_df.iloc[val_idx][base_feats].copy(),
                base_feats,
            )
        return train_df[base_feats].copy(), test_df[base_feats].copy(), base_feats

    other_cols = [c for c in base_feats if c != col]
    if tr_idx is not None:
        tr_df = train_df.iloc[tr_idx]
        val_df = train_df.iloc[val_idx]
        ref_df = tr_df
    else:
        tr_df = train_df
        val_df = test_df
        ref_df = train_df

    def _str_len(ser):
        return ser.astype(str).str.len()

    def _word_count(ser):
        return ser.astype(str).str.split().str.len().fillna(0).astype(int)

    def _freq_map(ser):
        return ser.astype(str).value_counts()

    if kind == "meta":
        # 列をやめて長さ・語数のみ
        feats = other_cols + ["movie_info_len", "movie_info_word_count"]
        if tr_idx is not None:
            X_tr = tr_df[other_cols].copy()
            X_val = val_df[other_cols].copy()
            X_tr["movie_info_len"] = _str_len(tr_df[col])
            X_tr["movie_info_word_count"] = _word_count(tr_df[col])
            X_val["movie_info_len"] = _str_len(val_df[col])
            X_val["movie_info_word_count"] = _word_count(val_df[col])
            return X_tr, X_val, feats
        X_train = train_df[other_cols].copy()
        X_test = test_df[other_cols].copy()
        X_train["movie_info_len"] = _str_len(train_df[col])
        X_train["movie_info_word_count"] = _word_count(train_df[col])
        X_test["movie_info_len"] = _str_len(test_df[col])
        X_test["movie_info_word_count"] = _word_count(test_df[col])
        return X_train, X_test, feats

    if kind == "freq_only":
        vc = _freq_map(ref_df[col])
        feats = other_cols + [f"{col}_freq"]
        def map_freq(ser):
            return ser.astype(str).map(vc).fillna(0).astype(int)
        if tr_idx is not None:
            X_tr = tr_df[other_cols].copy()
            X_val = val_df[other_cols].copy()
            X_tr[f"{col}_freq"] = map_freq(tr_df[col])
            X_val[f"{col}_freq"] = map_freq(val_df[col])
            return X_tr, X_val, feats
        X_train = train_df[other_cols].copy()
        X_test = test_df[other_cols].copy()
        X_train[f"{col}_freq"] = map_freq(train_df[col])
        X_test[f"{col}_freq"] = map_freq(test_df[col])
        return X_train, X_test, feats

    if kind == "add_freq":
        vc = _freq_map(ref_df[col])
        feats = other_cols + [col, f"{col}_freq"]
        def map_freq(ser):
            return ser.astype(str).map(vc).fillna(0).astype(int)
        if tr_idx is not None:
            X_tr = tr_df[other_cols + [col]].copy()
            X_val = val_df[other_cols + [col]].copy()
            X_tr[f"{col}_freq"] = map_freq(tr_df[col])
            X_val[f"{col}_freq"] = map_freq(val_df[col])
            return X_tr, X_val, feats
        X_train = train_df[other_cols + [col]].copy()
        X_test = test_df[other_cols + [col]].copy()
        X_train[f"{col}_freq"] = map_freq(train_df[col])
        X_test[f"{col}_freq"] = map_freq(test_df[col])
        return X_train, X_test, feats

    raise ValueError(f"Unknown kind: {kind}")

In [ ]:
# 各設定で CV を実行し、ベースとの差分を記録
preprocess_results = []
cv_auc_baseline_preprocess = None  # ベース（8列category）の CV

for cfg in PREPROCESS_EXPERIMENTS:
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(time_splits):
        X_tr, X_val, feats = build_X_for_config(train_work, test_work, cfg, tr_idx=tr_idx, val_idx=val_idx)
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
        pred_val = model.predict_proba(X_val)[:, 1]
        fold_scores.append(roc_auc_score(y_val, pred_val))
    cv_auc = np.mean(fold_scores)
    if cv_auc_baseline_preprocess is None:
        cv_auc_baseline_preprocess = cv_auc
    diff = cv_auc - cv_auc_baseline_preprocess
    preprocess_results.append({
        "設定": cfg["name"],
        "n_features": len(feats),
        "CV_AUC": round(cv_auc, 4),
        "差分(ベース比)": round(diff, 4),
    })
    print(f"  {cfg['name']}: CV_AUC={cv_auc:.4f}, 差分={diff:+.4f}")

In [ ]:
# 結果テーブル（ベースからどれだけ伸びたか）
df_preprocess = pd.DataFrame(preprocess_results)
display(df_preprocess)

# どの特徴量の前処理が効いているか（差分がプラスのもの）
improved = df_preprocess[df_preprocess["差分(ベース比)"] > 0].sort_values("差分(ベース比)", ascending=False)
if len(improved) > 0:
    print("\nベースより伸びた前処理（効いている特徴量）:")
    for _, row in improved.iterrows():
        if row["設定"] != "ベース（8列はcategory）":
            print(f"  + {row['設定']}: 差分 {row['差分(ベース比)']:+.4f}")
else:
    print("\nベースより伸びた前処理はありません（いずれもベース以下）。")

In [13]:
train.columns

Index(['ID', 'rotten_tomatoes_link', 'critic_name', 'top_critic',
       'publisher_name', 'review_date', 'movie_title', 'movie_info',
       'content_rating', 'genres', 'directors', 'authors', 'actors',
       'original_release_date', 'runtime', 'production_company', 'target',
       'review_year', 'review_month', 'review_dayofweek', 'release_year',
       'release_month', 'release_dayofweek', 'movie_age_days', 'genre_Drama',
       'genre_Comedy', 'genre_Action', 'genre_Mystery', 'genre_Fantasy',
       'genre_Romance', 'genre_Horror', 'genre_Documentary'],
      dtype='object')

In [16]:
train["release_year"]

0         2010.0
1         2010.0
2         2010.0
3         2010.0
4         2010.0
           ...  
653502    1979.0
653503    1979.0
653504    1979.0
653505    1979.0
653506    1979.0
Name: release_year, Length: 653507, dtype: float64

### 提出用: 拡張特徴で全 train 学習 → 予測

In [15]:
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X_ext, y)
final_pred = model_full.predict_proba(X_test_ext)[:, 1]

submission = pd.DataFrame({"ID": test_work["ID"], "target": final_pred})
submission.to_csv("submission_extended.csv", index=False)
print(f"Saved submission_extended.csv (rows: {len(submission):,}) [拡張特徴・全trainで学習]")

Saved submission_extended.csv (rows: 40,716) [拡張特徴・全trainで学習]
